# Hallucination check: manual review of parsed slides


In [ ]:
import json
import random
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
PARSED = ROOT / "data/parsed_NO_EDIT/machine_learning"
ANCHOR = ROOT / "data/eval/test_jsonfiles/anchor"
SLIDE_IMAGES = ROOT / "data/reference_slides/machine_learning"
OUT = ROOT / "data/eval/parsing"

LECTURES = [
    {
        "name": "SVM",
        "no_anchor": PARSED / "ML_5_svm/ML_5_svm_chunks.json",
        "anchor": ANCHOR / "ML_5_svm_chunks_docling_anchor.json",
    },
    {
        "name": "Neuronale Netze",
        "no_anchor": PARSED / "ML_9_neuronale_netze/ML_9_neuronale_netze_chunks.json",
        "anchor": ANCHOR / "ML_9_neuronale_netze_chunks_docling_anchor.json",
    },
]

GRAPHIC_PREFIX = "[GRAFIK]"

def is_graphic(paragraph):
    return paragraph.lstrip().startswith(GRAPHIC_PREFIX)

def reviewable_content(chunk):
    body = chunk.get("page_content", "").replace("\\n", "\n")
    paragraphs = re.split(r"\n\s*\n", body)
    return [p for p in paragraphs if p.strip() and not is_graphic(p)]

slides = {}
for lecture in LECTURES:
    parses_without_anchor = {
        chunk["id"]: chunk
        for chunk in json.loads(lecture["no_anchor"].read_text(encoding="utf-8"))
    }
    parses_with_anchor = {
        chunk["id"]: chunk
        for chunk in json.loads(lecture["anchor"].read_text(encoding="utf-8"))
    }

    for slide_id in sorted(parses_without_anchor.keys() & parses_with_anchor.keys()):
        chunk = parses_without_anchor[slide_id]
        page_number = chunk["page_numbers"][0]
        slides[slide_id] = {
            "lecture": lecture["name"],
            "image": SLIDE_IMAGES / chunk["lecture"] / f"page_{page_number}.png",
            "no_anchor": chunk,
            "anchor": parses_with_anchor[slide_id],
        }

eligible = [
    slide_id for slide_id, slide in slides.items()
    if reviewable_content(slide["no_anchor"]) and reviewable_content(slide["anchor"])
]

print("Slides:", len(slides))
print("Eligible (reviewable content beyond the title in both variants):", len(eligible))

## Sample

Simple random sample over all slides that both variants parsed. Fixed seed so the selection
stays reproducible. The variants are shown labelled, not blinded, so findings can be attributed
to a variant directly.

In [ ]:
SEED = 40
SAMPLE_SIZE = 30

random_generator = random.Random(SEED)

sample = sorted(random_generator.sample(sorted(eligible), SAMPLE_SIZE))

print(f"{len(sample)} slides in the sample:")
for slide_id in sample:
    print(f"  {slide_id}")

## Display

Slide image on the left, the two parse variants on the right. For each slide and variant, count
the statements the slide does not support, following the rules below.

In [ ]:
import base64
import html

from IPython.display import HTML, display

VARIANT_LABELS = {"no_anchor": "ohne Anker", "anchor": "mit Anker"}


def slide_image_tag(image_path):
    encoded_image = base64.b64encode(image_path.read_bytes()).decode()
    return f"<img src='data:image/png;base64,{encoded_image}' style='width:100%;border:1px solid #ccc'>"


def all_paragraphs(chunk):
    body = (chunk.get("title", "") + "\n\n" + chunk.get("page_content", "")).replace("\\n", "\n")
    return [p for p in re.split(r"\n\s*\n", body) if p.strip()]


def parse_block(variant, chunk):
    body = "".join(
        "<div style='white-space:pre-wrap;margin-bottom:10px'>" + html.escape(p) + "</div>"
        for p in all_paragraphs(chunk)
    )
    return (
        "<div style='flex:1;font-size:12px;line-height:1.45'>"
        f"<div style='font-weight:700;margin-bottom:6px'>{VARIANT_LABELS[variant]}</div>"
        + body
        + "</div>"
    )


blocks = []
for slide_id in sample:
    slide = slides[slide_id]
    header = (
        f"<h2 style='margin:2px 0'>{html.escape(slide_id)}</h2>"
        f"<div style='color:#888;font-size:12px'>{slide['lecture']}</div>"
    )
    parses = "".join(
        parse_block(variant, slide[variant]) for variant in ("no_anchor", "anchor")
    )
    row = (
        "<div style='display:flex;gap:16px;align-items:flex-start;margin-top:8px'>"
        "<div style='flex:1.1'>" + slide_image_tag(slide["image"]) + "</div>"
        "<div style='flex:1.4;display:flex;gap:16px'>" + parses + "</div>"
        "</div>"
    )
    blocks.append("<div style='border-top:2px solid #bbb;padding:14px 0'>" + header + row + "</div>")

display(HTML("".join(blocks)))

## My verdicts

Per slide and variant: the number of **hallucinations** — statements whose subject does not
appear on the slide at all (an invented term, fact, formula or code detail).

`[GRAFIK]` blocks are shown for context but are **not counted**. Both prompts describe graphics
from the image alone (the anchor text cannot capture images), so differences there are not
attributable to the anchor and would only measure VLM sampling noise.

Not counted either: omissions, wrong reading order and formatting. Those are fidelity defects,
not invented content, and are discussed separately.

In [ ]:
my_verdicts = [
    {"slide_id": "ML_5_svm_page_3", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_4", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_5", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_9", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_10", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_11", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_12", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_13", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_16", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_17", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_20", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_21", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_22", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_5_svm_page_24", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_1", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_2", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_3", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_6", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_9", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_10", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_13", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_16", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_17", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_20", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_21", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_25", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_27", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_28", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_33", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
    {"slide_id": "ML_9_neuronale_netze_page_37", "no_anchor_hallucination_count": 0, "anchor_hallucination_count": 0},
]

# Guard against a stale list after the sample changes.
rated_slide_ids = {verdict["slide_id"] for verdict in my_verdicts}
if rated_slide_ids != set(sample):
    raise ValueError(f"my_verdicts does not match the sample: {rated_slide_ids ^ set(sample)}")

print(f"{len(my_verdicts)} of {len(sample)} slides rated")

### Reasons

/

## Evaluation

In [ ]:
rows = []
for verdict in my_verdicts:
    slide_id = verdict["slide_id"]
    for variant in ("no_anchor", "anchor"):
        rows.append({
            "Folie": slide_id,
            "Vorlesung": slides[slide_id]["lecture"],
            "Variante": VARIANT_LABELS[variant],
            "Halluzinationen": verdict[f"{variant}_hallucination_count"],
        })

items = pd.DataFrame(rows)

# One row per slide and variant: pandas would otherwise truncate the middle of the table.
with pd.option_context("display.max_rows", None):
    display(items)

In [ ]:
summary = (
    items.groupby("Variante")
    .agg(
        Folien=("Folie", "count"),
        Halluzinationen=("Halluzinationen", "sum"),
        Halluzinierte_Folien=("Halluzinationen", lambda counts: int((counts > 0).sum())),
    )
    .reset_index()
    .rename(columns={"Halluzinierte_Folien": "Folien mit Halluzination"})
)

items.to_csv(OUT / "parsing_hallucination_items.csv", index=False, encoding="utf-8")
summary.to_csv(OUT / "parsing_hallucination_summary.csv", index=False, encoding="utf-8")
print("saved to", OUT)
display(summary.style.hide(axis="index"))

With ten slides per variant, report absolute counts rather than rates. The sample can only
reveal coarse differences; that limitation belongs in the discussion.